# **Start Section:**

In [1]:
!pip uninstall -y scikit-learn
!pip install scikit-learn==1.5.2
!pip install bayesian-optimization
!pip install optuna
!pip install catboost
!pip install gpboost
!pip install shap
!pip install ngboost
!pip install dask[dataframe]
!pip install torch seaborn
!pip install lightgbm
!pip install xgboost
!pip install lime
!pip install interpret
!pip install optunahub
!pip install cmaes
!pip install plotly kaleido
!pip install openpyxl
!pip install -U kaleido
!pip install properscoring
!pip install XlsxWriter
!pip install cp
!pip install torch skorch puncc

Found existing installation: scikit-learn 1.5.2
Uninstalling scikit-learn-1.5.2:
  Successfully uninstalled scikit-learn-1.5.2
  Using cached scikit_learn-1.5.2-cp312-cp312-win_amd64.whl.metadata (13 kB)
Using cached scikit_learn-1.5.2-cp312-cp312-win_amd64.whl (11.0 MB)


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from optuna.samplers import BaseSampler
from optuna.samplers import GridSampler
from optuna.samplers import TPESampler
from optuna.samplers import PartialFixedSampler
from optuna.samplers import CmaEsSampler
from optuna.samplers import QMCSampler
from optuna.samplers import NSGAIIISampler
from optuna.samplers import NSGAIISampler
from optuna.samplers import BruteForceSampler
from optuna.samplers import GPSampler
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json 
from mapie.subsample import Subsample
from mapie.regression import MapieRegressor
from deel.puncc.metrics import regression_sharpness, regression_mean_coverage
from deel.puncc.api.prediction import BasePredictor, DualPredictor
from deel.puncc.regression import SplitCP, CVPlus, CQR
from deel.puncc.plotting import plot_prediction_intervals
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from mapie.metrics import regression_coverage_score
from sklearn.model_selection import KFold
from PIL import Image as PImage

In [3]:
train_data_path = r"C:\Users\Danesh\Desktop\Concrete\train.csv"
test_data_path = r"C:\Users\Danesh\Desktop\Concrete\test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [4]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (189, 8)
First 5 rows of training data:
         C    mp     FA      CA       F       W_P    Adm    str
0  280.80  70.2  858.0  1183.0    0.00  0.450000  0.610  21.56
1  372.15   0.0  975.0   525.0   52.85  0.493081  7.000  34.00
2  360.00  75.0  975.0   525.0   40.00  0.509722  7.000  28.00
3  364.30   0.0  975.0   525.0  110.40  0.503706  7.000  42.00
4  315.00  31.5  780.0  1110.0    0.00  0.370000  5.355  25.82

Shape of test data: (95, 8)
First 5 rows of test data:
         C     mp     FA     CA      F       W_P   Adm   str
0  364.30    0.0  975.0  525.0  60.40  0.503706   7.0  34.0
1  344.30   75.0  975.0  525.0  55.40  0.532965   7.0  36.0
2  390.00    0.0  975.0  525.0  60.00  0.470513   7.0  36.0
3  352.15   75.0  975.0  525.0  47.85  0.521085   7.0  35.0
4  400.00  160.0  801.0  801.0  40.00  0.300000  10.3  44.6


In [5]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (189, 7)
Shape of y_train: (189,)
Shape of X_test: (95, 7)
Shape of y_test: (95,)


In [6]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[-1.56879184  0.95792014 -0.09685034  1.33138508 -0.85369002  0.16810471
  -0.08799752]
 [ 0.40300047 -0.7708921   0.73537613 -0.93459372  0.061075    0.78285857
  -0.0664621 ]
 [ 0.14074238  1.07612953  0.73537613 -0.93459372 -0.16134185  1.020321
  -0.0664621 ]
 [ 0.233558   -0.7708921   0.73537613 -0.93459372  1.05719093  0.93447436
  -0.0664621 ]
 [-0.83058388  0.00485698 -0.65166799  1.07999229 -0.85369002 -0.97347298
  -0.07200604]]

First five rows of normalized X_test:
[[ 0.233558   -0.7708921   0.73537613 -0.93459372  0.19175572  0.93447436
  -0.0664621 ]
 [-0.19814256  1.07612953  0.73537613 -0.93459372  0.1052122   1.35199213
  -0.0664621 ]
 [ 0.78829322 -0.7708921   0.73537613 -0.93459372  0.18483223  0.4608195
  -0.0664621 ]
 [-0.02870009  1.07612953  0.73537613 -0.93459372 -0.02546852  1.18246784
  -0.0664621 ]
 [ 1.0041435   3.1694207  -0.50229401  0.01587763 -0.16134185 -1.97235346
  -0.05534052]]


# **Functions:**

In [7]:
# Define the model classes
feature_names = ['C', 'mp', 'FA', 'CA', 'F', 'W_P', 'Adm']


In [8]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost'
    }
    
    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")
    
    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}
    
    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])
    
    # Return the best hyperparameters
    return best_entry_value['best_params']

In [9]:
def quantile_regression_print(predictions_df, file_path):
    # Extract actual values
    actual_values = predictions_df['Actual'].values

    # Debugging: Print the first few actual values
    print("First few actual values:", actual_values[:5])

    # Calculate coverage for the specific interval (0.05 to 0.95)
    lower_quantile = 0.05
    median_quantile = 0.5
    upper_quantile = 0.95

    lower_preds = predictions_df[lower_quantile].values
    median_preds = predictions_df[median_quantile].values
    upper_preds = predictions_df[upper_quantile].values

    in_interval = ((actual_values >= lower_preds) & (actual_values <= upper_preds))
    coverage = in_interval.mean()
    print(f"Coverage of 90% prediction interval: {coverage * 100:.2f}%")

    # Create a new Excel workbook and add a worksheet
    wb = Workbook()
    ws = wb.active
    ws.title = "quantile_regression_print"
    
    # Plot actual vs. all predicted quantiles
    num_points = len(predictions_df)
    indices = np.arange(num_points)

    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values[:num_points], label='Actual Concrete Strength', marker='o', linestyle='-', color='black')

    # Loop over all columns assumed to be quantile predictions
    for column in predictions_df.columns:
        if column != 'Actual':  # Skip the actual values column
            plt.plot(indices, predictions_df[column][:num_points],
                     label=f'Predicted Quantile {column}', linestyle='--')

    plt.xlabel('Sample Number')
    plt.ylabel('Concrete Strength')
    plt.title('Actual vs. Predicted Quantile Strengths')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)

    # Save the plot to a BytesIO object
    img_data = BytesIO()
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A1')

    # Plot actual values and specific prediction intervals (0.05, 0.5, 0.95)
    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values, label='Actual Strength', marker='o', linestyle='-', color='black')  # Connect points with a line
    plt.plot(indices, median_preds, label=f'Median Prediction ({median_quantile})', marker='x', linestyle='-', color='blue')
    plt.fill_between(
        indices,
        lower_preds,
        upper_preds,
        color='blue',
        alpha=0.5,
        label=f'Prediction Interval ({lower_quantile}-{upper_quantile})'
    )
    plt.xlabel('Sample Number')
    plt.ylabel(' Concrete Strength')
    plt.title('Actual Strength with Prediction Intervals')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1, 1))

    # Save the second plot to another BytesIO object
    img_data = BytesIO()
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the second image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A30')  # Adjust the cell position as needed

    # Save the workbook
    wb.save(file_path)

In [10]:
def evaluation_features_print(predictions_df, X_test, file_path):
    # Ensure X_test is a DataFrame
    if isinstance(X_test, np.ndarray):
        X_test = pd.DataFrame(X_test, columns=[f'Feature_{i}' for i in range(X_test.shape[1])])

    # Rename columns to match feature_names
    X_test.columns = feature_names

    # Get quantile columns from predictions_df
    # Assuming that quantile columns are all columns except 'Actual'
    quantile_columns = [col for col in predictions_df.columns if col != 'Actual']
    quantiles = quantile_columns  # Assuming the column names represent quantiles

    # Check if the Excel file already exists
    if os.path.exists(file_path):
        # Load the existing workbook
        wb = load_workbook(file_path)
        # Create a new worksheet
        ws = wb.create_sheet(title="New Evaluation Features")
    else:
        # Create a new workbook and add a worksheet
        wb = Workbook()
        ws = wb.active
        ws.title = "evaluation_features_print"

    image_row = 1  # Start row for placing images

    # Loop through each feature and create a plot
    for feature_to_plot in feature_names:
        # Extract feature values corresponding to the test set
        feature_values = X_test[feature_to_plot]

        # Create a DataFrame for Plotly
        plotly_df = predictions_df.copy()
        plotly_df[feature_to_plot] = feature_values.values

        # Melt the DataFrame to long format
        melted_df = plotly_df.melt(
            id_vars=[feature_to_plot, 'Actual'],
            value_vars=quantiles,
            var_name='Quantile',
            value_name='Prediction'
        )

        # Optional: Sort the data for better visualization
        melted_df.sort_values(by=feature_to_plot, inplace=True)

        # Plot
        fig = px.line(
            melted_df,
            x=feature_to_plot,
            y='Prediction',
            color='Quantile',
            title=f'Predicted Quantiles vs. {feature_to_plot}',
            labels={'Prediction': 'Strength'}
        )

        # Add actual values as scatter points
        fig.add_scatter(
            x=plotly_df[feature_to_plot],
            y=plotly_df['Actual'],
            mode='markers',
            name='Actual Strength'
        )

        # Save the plot to a BytesIO object
        img_data = BytesIO()
        fig.write_image(img_data, format='png')
        img_data.seek(0)

        # Insert the image into the Excel sheet
        img = Image(img_data)
        img.anchor = f'A{image_row}'
        ws.add_image(img)

        # Increase image_row to place the next image below this one
        image_row += 20  # Adjust as needed to avoid overlap

    # Save the workbook
    wb.save(file_path)


In [11]:
def evaluate_uncertainity_matrix(predictions_df, quantiles, excel_file_path, model_name='Model'):

    # # Set plotting style
    # sns.set(style="whitegrid")

    # Extract the actual target values
    y_true = predictions_df['Actual']

    # Open existing Excel file
    wb = load_workbook(filename=excel_file_path)

    # Create a unique sheet name
    def get_unique_sheet_name(wb, base_name):
        sheet_name = base_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base_name}_{i}"
            i += 1
        return sheet_name

    sheet_title = get_unique_sheet_name(wb, 'evaluate_uncertainity_matrix')

    # Create a new worksheet for the plots
    ws = wb.create_sheet(title=sheet_title)

    # Positions to place images
    image_positions = ['A1', 'A25', 'A49', 'A73']  # Adjust as needed

    ### 1. Validity / Coverage ###

    # Compute empirical coverage for each quantile
    empirical_quantiles = []
    for q in quantiles:
        coverage = (predictions_df[q] >= y_true).mean()
        empirical_quantiles.append(coverage)

    # Plot theoretical quantiles vs empirical quantiles
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=quantiles, y=quantiles, color="magenta", linestyle='--', linewidth=2, label="Ideal")
    sns.lineplot(x=quantiles, y=empirical_quantiles, color="blue", linewidth=2, label="Empirical")
    sns.scatterplot(x=quantiles, y=empirical_quantiles, color="blue", s=50)
    plt.xlabel("Nominal Quantile Levels")
    plt.ylabel("Empirical Quantile Levels")
    plt.title(f"Validity / Coverage - {model_name}")
    plt.legend()

    # Save plot to buffer
    buffer = io.BytesIO()
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add the image to the worksheet
    ws.add_image(img, image_positions[0])

    ### 2. Sharpness / Interval Length ###

    # Define coverage levels to evaluate
    coverage_levels = [0.6, 0.8, 0.9]
    average_interval_lengths = []

    # Compute average interval lengths for each coverage level
    for c in coverage_levels:
        lower_q = (1 - c) / 2
        upper_q = 1 - lower_q
        # Find the nearest quantiles in our quantiles list
        lower_quantile = min(quantiles, key=lambda x: abs(x - lower_q))
        upper_quantile = min(quantiles, key=lambda x: abs(x - upper_q))

        # Compute interval length for each observation
        interval_length = predictions_df[upper_quantile] - predictions_df[lower_quantile]
        avg_interval_length = interval_length.mean()
        average_interval_lengths.append(avg_interval_length)
        print(f'Coverage level: {c*100:.0f}%, Interval: [{lower_quantile}, {upper_quantile}], Average Interval Length: {avg_interval_length:.4f}')

    # Plot coverage levels vs average interval lengths
    plt.figure(figsize=(10, 6))
    plt.plot([c*100 for c in coverage_levels], average_interval_lengths, marker='o')
    plt.xlabel('Coverage Level (%)')
    plt.ylabel('Average Interval Length')
    plt.title(f'Sharpness / Interval Length - {model_name}')
    # plt.grid(True)

    # Save plot to buffer
    buffer = io.BytesIO()
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to worksheet
    ws.add_image(img, image_positions[1])

    ### 3. Negative Log-Likelihood (NLL) ###

    # Use the 5th and 95th percentiles to estimate standard deviation
    if 0.05 in quantiles and 0.95 in quantiles:
        z_lower = norm.ppf(0.05)  # z-score for 5% quantile (~ -1.6449)
        z_upper = norm.ppf(0.95)  # z-score for 95% quantile (~ +1.6449)

        # Extract quantile predictions
        q_lower = predictions_df[0.05]
        q_upper = predictions_df[0.95]

    elif 0.1 in quantiles and 0.9 in quantiles:
        z_lower = norm.ppf(0.1)  # z-score for 10% quantile (~ -1.2816)
        z_upper = norm.ppf(0.9)  # z-score for 90% quantile (~ +1.2816)

        # Extract quantile predictions
        q_lower = predictions_df[0.1]
        q_upper = predictions_df[0.9]
    else:
        raise ValueError("Required quantiles for NLL estimation not found in quantiles list.")

    # Use median as the mean estimate
    if 0.5 in quantiles:
        q_median = predictions_df[0.5]
    else:
        # If 0.5 quantile is not available, use the middle quantile
        q_median = predictions_df[quantiles[len(quantiles)//2]]

    # Estimate standard deviation for each observation
    std_estimates = (q_upper - q_lower) / (z_upper - z_lower)

    # Ensure standard deviations are positive and non-zero
    std_estimates = std_estimates.clip(lower=1e-6)

    # Extract mean estimates (median predictions)
    mean_estimates = q_median

    # Compute Negative Log-Likelihood for each observation
    nll = -norm.logpdf(y_true, loc=mean_estimates, scale=std_estimates)

    # Compute average NLL
    average_nll = nll.mean()
    print(f'Average Negative Log-Likelihood (NLL): {average_nll:.4f}')

    # Plot ECDF of NLL values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(nll, color='blue', linewidth=2)
    plt.xlabel('Negative Log-Likelihood (NLL)')
    plt.ylabel('ECDF')
    plt.title(f'NLL ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to worksheet
    ws.add_image(img, image_positions[2])

    ### 4. Continuous Ranked Probability Score (CRPS) ###

    # Prepare ensemble of quantile predictions for each observation
    ensemble_predictions = predictions_df[quantiles].to_numpy()

    # Ensure quantile predictions are sorted for each observation
    ensemble_predictions.sort(axis=1)

    # Compute CRPS for each observation using ensemble predictions
    crps_values = ps.crps_ensemble(y_true, ensemble_predictions)

    # Compute average CRPS
    average_crps = crps_values.mean()
    print(f'Average Continuous Ranked Probability Score (CRPS): {average_crps:.4f}')

    # Plot ECDF of CRPS values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(crps_values, color='green', linewidth=2)
    plt.xlabel('Continuous Ranked Probability Score (CRPS)')
    plt.ylabel('ECDF')
    plt.title(f'CRPS ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to worksheet
    ws.add_image(img, image_positions[3])

    # Save the workbook
    wb.save(excel_file_path)

    # Return a dictionary of results
    results = {
        'empirical_quantiles': empirical_quantiles,
        'average_interval_lengths': average_interval_lengths,
        'coverage_levels': coverage_levels,
        'average_nll': average_nll,
        'average_crps': average_crps,
        'nll_values': nll,
        'crps_values': crps_values
    }

    return results

In [12]:
def calibrate_and_plot_intervals(predictions_df, excel_file_path, alpha=0.1, max_samples=1000, random_seed=42):
    # Extract necessary columns
    lower = predictions_df[0.05].fillna(0)
    upper = predictions_df[0.95].fillna(0)
    pred = predictions_df[0.5].fillna(0)
    labels = predictions_df['Actual'].fillna(0)

    # Problem setup
    total_samples = labels.shape[0]
    n = min(max_samples, total_samples)  # Ensure n does not exceed the number of samples

    # Ensure there are enough samples for both calibration and validation
    if total_samples <= n:
        n = total_samples // 2

    # Split the data into calibration and validation sets
    np.random.seed(random_seed)  # Set the random seed for reproducibility
    idx = np.array([1] * n + [0] * (total_samples - n)) > 0
    np.random.shuffle(idx)
    cal_labels, val_labels = labels[idx], labels[~idx]
    cal_upper, val_upper = upper[idx], upper[~idx]
    cal_lower, val_lower = lower[idx], lower[~idx]
    cal_pred, val_pred = pred[idx], pred[~idx]

    # Calculate uncertainty intervals
    cal_U = cal_upper - cal_lower
    val_U = val_upper - val_lower

    # Avoid division by zero by replacing zero intervals with a small number
    cal_U[cal_U == 0] = np.finfo(float).eps
    val_U[val_U == 0] = np.finfo(float).eps

    # Get scores
    cal_scores = np.abs(cal_pred - cal_labels) / cal_U

    # Debugging: Check for NaN in scores
    if np.isnan(cal_scores).any():
        print("NaN values found in cal_scores. Check data integrity.")

    # Get the score quantile
    qhat = np.quantile(cal_scores, np.ceil((n + 1) * (1 - alpha)) / n, interpolation='higher')

    # Deploy (output=lower and upper adjusted quantiles)
    prediction_sets = [val_pred - val_U * qhat, val_pred + val_U * qhat]

    # Calculate empirical coverage (before and after calibration)
    prediction_sets_uncalibrated = [val_lower, val_upper]
    empirical_coverage_uncalibrated = ((val_labels >= prediction_sets_uncalibrated[0]) & (val_labels <= prediction_sets_uncalibrated[1])).mean() * 100
    print(f"The empirical coverage before calibration is: {empirical_coverage_uncalibrated:.2f}%")

    empirical_coverage = ((val_labels >= prediction_sets[0]) & (val_labels <= prediction_sets[1])).mean() * 100
    print(f"The empirical coverage after calibration is: {empirical_coverage:.2f}%")

    # Create a DataFrame with all the necessary data
    data = pd.DataFrame({
        'Index': np.arange(len(val_labels)),
        'Uncalibrated Lower': val_lower,
        'Uncalibrated Upper': val_upper,
        'Calibrated Lower': prediction_sets[0],
        'Calibrated Upper': prediction_sets[1],
        'True Label': val_labels,
        'Predicted Value': val_pred
    })

    plt.figure(figsize=(14, 8))

    # Plot the uncalibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Uncalibrated Lower'],
        data['Uncalibrated Upper'],
        color='blue',
        alpha=0.5,
        label='Uncalibrated Prediction Interval'
    )

    # Plot the calibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Calibrated Lower'],
        data['Calibrated Upper'],
        color='lightgreen',
        alpha=0.5,
        label='Calibrated Prediction Interval'
    )

    # Plot the true labels as points and connect them with a line
    plt.plot(
        data['Index'],
        data['True Label'],
        'o-',
        color='red',
        markersize=4,
        label='Actual Value'
    )

    # Plot the predicted values as points and connect them with a line
    plt.plot(
        data['Index'],
        data['Predicted Value'],
        'o-',
        color='black',
        markersize=4,
        label='Predicted Value'
    )

    plt.xlabel('Sample Number')
    plt.ylabel('Concrete Strength')
    plt.title('Calibrated and Uncalibrated Prediction Intervals with True and Predicted Values')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()

    # Save the plot as an image
    image_path = 'plot.png'
    plt.savefig(image_path)
    plt.close()

    # Load the existing Excel file
    workbook = load_workbook(excel_file_path)

    # Create a new sheet for the plot
    new_sheet_name = "calibrate_and_plot_intervals"
    workbook.create_sheet(title=new_sheet_name)

    # Insert the image into the new sheet
    sheet = workbook[new_sheet_name]
    img = Image(image_path)
    sheet.add_image(img, 'A1')

    # Save the workbook
    workbook.save(excel_file_path)

In [13]:
def plot_conformal_intervals_to_excel(predictions_df, file_path, alpha=0.1):
    """
    Plots conformal prediction intervals and their coverage, and saves the plots to an Excel file.

    Parameters:
    - predictions_df: DataFrame containing prediction quantiles and actual values.
    - file_path: Path to the Excel file where plots will be saved.
    - alpha: Desired coverage level (default is 0.1).
    """
    # Extract mean prediction and uncertainty
    mean_prediction = predictions_df[0.5]
    uncertainty = predictions_df[0.95] - predictions_df[0.05]
    actual_values = predictions_df['Actual']

    # Calculate conformal scores
    conformal_scores = np.abs(mean_prediction - actual_values) / uncertainty

    # Function to find weighted quantile
    def weighted_quantile(values, quantile, sample_weight=None):
        values = np.array(values)
        if sample_weight is None:
            sample_weight = np.ones(len(values))
        sorter = np.argsort(values)
        values, sample_weight = values[sorter], sample_weight[sorter]
        weighted_quantiles = np.cumsum(sample_weight) - 0.5 * sample_weight
        weighted_quantiles /= np.sum(sample_weight)
        return np.interp(quantile, weighted_quantiles, values)

    # Calculate weighted quantile
    weights = np.ones_like(conformal_scores)  # Uniform weights for simplicity
    quantile = weighted_quantile(conformal_scores, 1 - alpha, sample_weight=weights)

    # Calculate prediction intervals
    lower_bound = mean_prediction - quantile * uncertainty
    upper_bound = mean_prediction + quantile * uncertainty

    # Naive conformal prediction
    naive_quantile = np.quantile(conformal_scores, 1 - alpha)
    naive_lower_bound = mean_prediction - naive_quantile * uncertainty
    naive_upper_bound = mean_prediction + naive_quantile * uncertainty

    # Coverage calculation
    weighted_coverage_points = (actual_values >= lower_bound) & (actual_values <= upper_bound)
    naive_coverage_points = (actual_values >= naive_lower_bound) & (actual_values <= naive_upper_bound)

    # Visualization using Plotly
    fig = go.Figure()

    # Plot Actual Values
    fig.add_trace(go.Scatter(
        x=predictions_df.index,
        y=actual_values,
        mode='lines',
        name='Actual',
        line=dict(color='black', width=2)
    ))

    # Plot Mean Prediction
    fig.add_trace(go.Scatter(
        x=predictions_df.index,
        y=mean_prediction,
        mode='lines',
        name='Mean Prediction',
        line=dict(color='darkorange', width=2)
    ))

    # Plot Weighted Prediction Intervals
    fig.add_trace(go.Scatter(
        x=predictions_df.index,
        y=lower_bound,
        fill=None,
        mode='lines',
        line=dict(color='blue', width=0),
        showlegend=False
    ))
    fig.add_trace(go.Scatter(
        x=predictions_df.index,
        y=upper_bound,
        fill='tonexty',
        mode='lines',
        line=dict(color='blue', width=0),
        name='Weighted Interval',
        fillcolor='rgba(0, 0, 255, 0.3)'
    ))

    # Plot Naive Prediction Intervals
    fig.add_trace(go.Scatter(
        x=predictions_df.index,
        y=naive_lower_bound,
        fill=None,
        mode='lines',
        line=dict(color='red', width=0),
        showlegend=False
    ))
    fig.add_trace(go.Scatter(
        x=predictions_df.index,
        y=naive_upper_bound,
        fill='tonexty',
        mode='lines',
        line=dict(color='red', width=0),
        name='Naive Interval',
        fillcolor='rgba(255, 0, 0, 0.3)'
    ))

    # Update layout for better visualization
    fig.update_layout(
        title='Conformal Prediction Intervals',
        xaxis_title='Sample Number',
        yaxis_title='Concrete Strength',
        xaxis=dict(
            showline=True,
            showgrid=False,
            showticklabels=True,
            linecolor='black',
            linewidth=2,
            ticks='outside',
            tickfont=dict(
                family='Arial',
                size=12,
                color='black',
            ),
        ),
        yaxis=dict(
            showline=True,
            showgrid=False,
            showticklabels=True,
            linecolor='black',
            linewidth=2,
            ticks='outside',
            tickfont=dict(
                family='Arial',
                size=12,
                color='black',
            ),
        ),
        legend=dict(
            x=1.05,
            y=1,
            xanchor='left',
            yanchor='top',
            traceorder='normal',
            font=dict(
                family='Arial',
                size=12,
                color='black',
            ),
            bgcolor='rgba(255, 255, 255, 0)',
            bordercolor='black',
            borderwidth=1
        ),
        plot_bgcolor='white',
        width=800,   # Plot width
        height=400,  # Plot height
        margin=dict(
            l=60,    # left margin
            r=150,   # increased right margin to accommodate legend
            b=50,    # bottom margin
            t=50     # top margin
        ),
    )

    # Save the plot to a BytesIO object with adjusted dimensions
    img_data = BytesIO()
    fig.write_image(img_data, format='png', width=800, height=400, scale=1)
    img_data.seek(0)

    # Check if the Excel file already exists
    if os.path.exists(file_path):
        # Load the existing workbook
        wb = load_workbook(file_path)
        # Create a new worksheet
        ws = wb.create_sheet(title="plot_conformal_intervals_to_excel")
    else:
        # Create a new workbook and add a worksheet
        wb = Workbook()
        ws = wb.active
        ws.title = "plot_conformal_intervals_to_excel"

    # Insert the image into the Excel sheet
    img = Image(img_data)
    img.anchor = 'A1'
    ws.add_image(img)

    # Adjust the column widths to accommodate the image
    ws.column_dimensions['A'].width = 30

    # Save the workbook
    wb.save(file_path)

In [14]:
class validate:
    @staticmethod
    def coverage(int_pred, y_test):
        return {strat_name: np.mean((y_test.to_numpy() >= int_pred[strat_name][:, 0]) & (y_test.to_numpy() <= int_pred[strat_name][:, 1])) for strat_name in int_pred}
    @staticmethod
    def width(int_pred): # Corrected: Removed y_test argument as it's not defined and not used
        return {strat_name: np.mean(int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]) for strat_name in int_pred}
    @staticmethod
    def rmse(y_pred, y_test):
        return {strat_name: np.sqrt(mean_squared_error(y_test, y_pred[strat_name])) for strat_name in y_pred}
    @staticmethod
    def cwc(int_pred, y_test, miscoverage):
        widths = validate.width(int_pred)
        coverages = validate.coverage(int_pred, y_test)
        return {strat_name: widths[strat_name] * (1 + (coverages[strat_name] < (1 - miscoverage)) * (1 - coverages[strat_name])) for strat_name in int_pred}

    @staticmethod
    def cond_coverage(int_pred, y_test, num_bins=10):
        cond_coverages = {}
        for strat_name in int_pred:
            widths = int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]
            bin_edges = np.histogram_bin_edges(widths, bins=num_bins)
            digitized_widths = np.digitize(widths, bin_edges)
            strategy_cond_coverage = []
            for bin_num in range(1, num_bins + 1):
                indices_in_bin = np.where(digitized_widths == bin_num)[0]
                if len(indices_in_bin) > 0:
                    bin_coverage = np.mean((y_test.to_numpy()[indices_in_bin] >= int_pred[strat_name][indices_in_bin, 0]) & (y_test.to_numpy()[indices_in_bin] <= int_pred[strat_name][indices_in_bin, 1]))
                    strategy_cond_coverage.append(bin_coverage)
                else:
                    strategy_cond_coverage.append(np.nan) # or handle empty bins as needed
            cond_coverages[strat_name] = strategy_cond_coverage
        return cond_coverages


class visualize:
    @staticmethod
    def coverage(int_pred, y_test):
        return {strat_name: np.mean((y_test.to_numpy() >= int_pred[strat_name][:, 0]) & (y_test.to_numpy() <= int_pred[strat_name][:, 1])) for strat_name in int_pred}
    @staticmethod
    def width(int_pred):
        return {strat_name: np.mean(int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]) for strat_name in int_pred}
    @staticmethod
    def rmse(y_pred, y_test):
        return {strat_name: np.sqrt(mean_squared_error(y_test, y_pred[strat_name])) for strat_name in y_pred}
    @staticmethod
    def cwc(int_pred, y_test, miscoverage):
        widths = visualize.width(int_pred)
        coverages = visualize.coverage(int_pred, y_test)
        return {strat_name: widths[strat_name] * (1 + (coverages[strat_name] < (1 - miscoverage)) * (1 - coverages[strat_name])) for strat_name in int_pred}
    @staticmethod
    def goodness(y_true, y_pred, y_pred_low, y_pred_up, coverage, width, rmse, cwc, ax=None, title="Goodness Plot"):
        if ax is None:
            fig, ax = plt.subplots()
        n_samples = len(y_true)
        y_true_np = y_true.to_numpy().ravel() # Ensure y_true is numpy array and flattened

        # Determine points inside and outside the interval
        inside_interval = (y_true_np >= y_pred_low) & (y_true_np <= y_pred_up)
        outside_interval = ~inside_interval

        # Plot points inside the interval in green
        ax.scatter(y_true_np[inside_interval], y_pred[inside_interval], color='green', s=10, label='Inside Interval')
        # Plot points outside the interval in orange
        ax.scatter(y_true_np[outside_interval], y_pred[outside_interval], color='orange', s=10, label='Outside Interval')

        # Plot all ground truth values as blue crosses
        ax.scatter(y_true_np, y_true_np, color='blue', marker='x', s=30, linewidths=0.7, label='Ground Truth Values')


        ax.vlines(y_true_np, y_pred_low, y_pred_up, color='gray', alpha=0.5, label='Prediction Intervals')
        ax.plot([min(y_true_np), max(y_true_np)], [min(y_true_np), max(y_true_np)], linestyle='--', color='blue', label='Ideal Prediction Line') # Changed label for clarity
        ax.set_xlabel('Ground Truth')
        ax.set_ylabel('Predictions')
        ax.set_title(f'{title}\nCoverage: {coverage*100:.2f}%, Width: {width:.2f}, RMSE: {rmse:.2f}, CWC: {cwc:.2f}')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax

    @staticmethod
    def width_size_occurrence(int_pred, train_intervals, num_bins=10, ax=None, x_lim=None, title="Width Size Occurrence"):
        if ax is None:
            fig, ax = plt.subplots()

        # Calculate interval widths for test and train sets
        widths_test = np.abs(int_pred[:, 1] - int_pred[:, 0])
        widths_train = np.abs(train_intervals[:, 1] - train_intervals[:, 0])

        # Plot histograms for both train and test widths
        ax.hist(widths_test, bins=num_bins, alpha=0.6, label='Test Interval Widths', color='red')
        ax.hist(widths_train, bins=num_bins, alpha=0.4, label='Train Interval Widths', color='blue')


        if x_lim is not None:
            ax.set_xlim(x_lim)
        ax.set_xlabel('Interval Width')
        ax.set_ylabel('Occurrence (Frequency)')
        ax.set_title(f'{title} - Interval Width Occurrence')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax

    @staticmethod
    def coverage_by_width(y_test, int_pred, miscoverage, cond_coverages, num_bins=10, ax=None, title="Coverage vs Width"):
        if ax is None:
            fig, ax = plt.subplots()

        widths = int_pred[:, 1] - int_pred[:, 0]
        bin_edges = np.histogram_bin_edges(widths, bins=num_bins)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        bin_width = bin_edges[1] - bin_edges[0]

        ax.bar(bin_centers, cond_coverages, width=bin_width, color='blue', alpha=0.7, edgecolor='black', linewidth=0.8, label='Conditional Coverage')
        ax.axhline(1 - miscoverage, color='red', linestyle='--', label=f'Nominal Coverage ({(1 - miscoverage)*100:.0f}%)')

        ax.set_xlabel('Interval Width')
        ax.set_ylabel('Conditional Coverage')
        ax.set_title(f'{title} - Coverage vs Interval Width')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax



def conformal_predictions_MAPIE(model_class, best_params, X_train, y_train, X_test, y_test, model_name, excel_file_path):
    """
    Train and plot conformal prediction intervals using different strategies and save the plots to an Excel file.

    Parameters:
    - model_class: The class of the model to be used (e.g., LGBMRegressor).
    - best_params: Dictionary of best parameters for the model.
    - X_train: Training features.
    - y_train: Training target.
    - X_test: Test features.
    - y_test: Test target.
    - model_name: Name of the model for labeling purposes.
    - excel_file_path: Path to the Excel file where the plots will be saved.
    """
    # Set random seed for reproducibility
    SEED: int = 123
    np.random.seed(SEED)

    # Define miscoverage
    MISCOVERAGE: float = 0.06  # MISCOVERAGE = alpha # CONFIDENCE = 1-alpha
    warnings.filterwarnings("ignore")  # to suppress them

    # Create output directory
    os.makedirs('output/regression', exist_ok=True)

    # Define strategies and parameters
    strategies_names = {
        'SCP': 'Split Conformal Prediction',
        'CV+': 'Cross Validation +',
        'J+aB': 'Jackknife+ after Bootstrapping',
    }
    strategies_params = {
        'SCP': {'method': 'base', 'cv': "split"},
        'CV+': {'method': "plus", 'cv': 10},
        'J+aB': {'method': "minmax", 'cv': Subsample(n_resamplings=50)},
    }

    # Define base estimators
    base_estimators = {
        strat_name: model_class(**best_params) for strat_name in strategies_names
    }

    # Train strategies using MAPIE
    y_pred, int_pred = {}, {}
    y_pred_train, int_pred_train = {}, {} # Store train predictions and intervals
    for strat_name, base_estimator in base_estimators.items():
        mapie = MapieRegressor(base_estimator, **strategies_params[strat_name])
        mapie.fit(X_train, y_train)
        y_pred[strat_name], int_pred[strat_name] = mapie.predict(X_test, alpha=MISCOVERAGE)
        y_pred_train[strat_name], int_pred_train[strat_name] = mapie.predict(X_train, alpha=MISCOVERAGE) # Predict on train set

    # Calculate metrics
    coverages: dict = validate.coverage(int_pred, y_test)
    widths: dict = validate.width(int_pred) # Corrected line: Removed y_test
    rmse_vals: dict = validate.rmse(y_pred, y_test)
    cwc_vals: dict = validate.cwc(int_pred, y_test, MISCOVERAGE)


    # Print metrics and prepare for plotting
    for strat_name in strategies_names.keys():
        if strat_name in y_pred:
            # Calculate RMSE (already calculated in validate.rmse, using rmse_vals now)
            rmse = rmse_vals[strat_name]

            # Calculate interval width (already calculated in validate.width, using widths now)
            interval_width = widths[strat_name]

            # Calculate coverage (already calculated in validate.coverage, using coverages now)
            coverage = coverages[strat_name]

            # Calculate CWC (already calculated in validate.cwc, using cwc_vals now)
            cwc = cwc_vals[strat_name]

            # Calculate SSC (Symmetric Scoring Criterion)
            ssc = np.mean((y_test.to_numpy() - y_pred[strat_name])**2 + (int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0])**2)

            print(f"Strategy: {strategies_names[strat_name]}")
            print(f"  RMSE: {rmse:.4f}")
            print(f"  Interval Width: {interval_width:.4f}")
            print(f"  Coverage: {coverage * 100:.2f}%")
            print(f"  CWC: {cwc:.4f}")
            print(f"  SSC: {ssc:.4f}")
            print()

    # Plot the results for the test set
    _strategies = [_s for _s in strategies_names.keys() if _s != 'SCP']
    NUM_BINS: int = 10
    n_figs: int = len(strategies_names) * 3 + 1 + len(_strategies) # Strategy plots + Goodness plots + Width plots + Comparison plot + Coverage vs Width plots
    fig, axs = plt.subplots(nrows=n_figs, figsize=(10, 5 * n_figs))

    plot_index = 0 # Keep track of the current plot index

    # Individual strategy plots (original plots)
    for _i, strat_name in enumerate(strategies_names.keys()):
        if strat_name in y_pred:
            # Predicted intervals
            _y_pred_low = int_pred[strat_name][:, 0].ravel()
            _y_pred_up = int_pred[strat_name][:, 1].ravel()

            # Visualize
            sample_indices = np.arange(len(X_test))
            axs[plot_index].scatter(sample_indices, y_test, label='Test data', color='blue', s=10, alpha=0.7)
            axs[plot_index].fill_between(sample_indices, _y_pred_low, _y_pred_up, color='gray', alpha=0.5, label='Prediction interval')
            axs[plot_index].plot(sample_indices, y_pred[strat_name], color='red', label='Predicted mean', linewidth=1)
            axs[plot_index].set_title(f'{strategies_names[strat_name]} ({model_name})')
            axs[plot_index].set_xlabel('Sample Number')
            axs[plot_index].set_ylabel('Concrete strength')
            axs[plot_index].legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
            plot_index += 1

        # Combined plot for comparison
    comparison_ax = axs[plot_index]
    sample_indices = np.arange(len(X_test))
    comparison_ax.scatter(sample_indices, y_test, label='Test data', color='blue', s=10, alpha=0.7)
    for strat_name in y_pred:
        _y_pred_low = int_pred[strat_name][:, 0].ravel()
        _y_pred_up = int_pred[strat_name][:, 1].ravel()
        comparison_ax.fill_between(sample_indices, _y_pred_low, _y_pred_up, alpha=0.3, label=f'{strat_name} interval')
        comparison_ax.plot(sample_indices, y_pred[strat_name], label=f'{strat_name} mean', linewidth=1)

    comparison_ax.set_title(f'Comparison of Strategies ({model_name})')
    comparison_ax.set_xlabel('Sample Number')
    comparison_ax.set_ylabel('Concrete Strength')
    comparison_ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plot_index += 1
    
    # Goodness plots
    for _i, strat_name in enumerate(strategies_names.keys()):
        if strat_name in y_pred:
            _y_pred_low_goodness = int_pred[strat_name][:, 0].ravel()
            _y_pred_up_goodness = int_pred[strat_name][:, 1].ravel()

            axs[plot_index] = visualize.goodness(
                y_test, y_pred[strat_name],
                _y_pred_low_goodness,
                _y_pred_up_goodness,
                coverages[strat_name],
                widths[strat_name],
                rmse_vals[strat_name],
                cwc_vals[strat_name],
                ax=axs[plot_index],
                title=f'{strategies_names[strat_name]} Goodness ({model_name})', # Removed subsample
            )
            axs[plot_index].set_ylabel('Predicted Concrete Strength')
            plot_index += 1

    # Width Occurrence plots
    _x_max = (1+1e-3) * np.max([np.abs(int_pred[strat_name][:, 0] - int_pred[strat_name][:, 1]) for strat_name in strategies_names.keys() if strat_name in int_pred])
    _x_min = np.min([np.abs(int_pred[strat_name][:, 0] - int_pred[strat_name][:, 1]) for strat_name in strategies_names.keys() if strat_name in int_pred])

    for _i, strat_name in enumerate(strategies_names.keys()):
        if strat_name in y_pred:
            axs[plot_index] = visualize.width_size_occurrence(
                int_pred[strat_name],
                train_intervals=int_pred_train[strat_name],
                num_bins=10,
                ax=axs[plot_index],
                x_lim=[_x_min, _x_max],
                title=f'{strategies_names[strat_name]} Width Occurrence',
            )
            axs[plot_index].set_ylabel('Frequency')
            plot_index += 1

    # Coverage vs Width plots
    cond_coverages: dict = validate.cond_coverage(int_pred, y_test, num_bins=NUM_BINS)
    for _i, _strat in enumerate(_strategies):
        axs[plot_index] = visualize.coverage_by_width(
            y_test, int_pred[_strat], MISCOVERAGE,
            cond_coverages[_strat],
            num_bins=NUM_BINS,
            ax=axs[plot_index],
            title=f'{strategies_names[_strat]} Coverage vs Width',
        )
        plot_index += 1

    plt.tight_layout()

    # Save plots to Excel
    with io.BytesIO() as buf:
        plt.savefig(buf, format='png')
        buf.seek(0)
        img = Image(PImage.open(buf)) # Use PIL to open from BytesIO

        # Load the workbook and add a new sheet
        workbook = load_workbook(excel_file_path)
        sheet_name = 'conformal_predictions_MAPIE'
        if sheet_name in workbook.sheetnames:
            sheet_name += '_new'
        worksheet = workbook.create_sheet(title=sheet_name)

        # Add the image to the worksheet
        worksheet.add_image(img, 'A1')

        # Save the workbook
        workbook.save(excel_file_path)

    plt.close()

In [15]:
def conformal_predictions_PUNCC(X_train, y_train, X_test, y_test, best_scores_autosampler, model_class, excel_file_path=None, model_params=None, alpha=0.1):
    # Determine the default parameters for the model, or use provided ones
    if model_params is None:
        if model_class.__name__ == 'LGBMRegressor':
            model_params = get_best_model_params(best_scores_autosampler, 'LightGBM')
        else:
            model_params = {}

    # Initialize and train the model using the provided class and parameters
    try:
        model = model_class(**model_params)
    except TypeError as e:
        print(f"Error initializing model {model_class.__name__}: {e}")
        return

    model.fit(X_train, y_train)

    def evaluate_cp(X_test, y_test, model_cp, alpha):
        y_pred, y_pred_lower, y_pred_upper = model_cp.predict(X_test, alpha=alpha)
        sharpness = regression_sharpness(y_pred_lower, y_pred_upper)
        coverage = regression_mean_coverage(y_test, y_pred_lower, y_pred_upper)
        return sharpness, coverage
    
    # Wrap the model in a BasePredictor
    base_predictor = BasePredictor(model, is_trained=True)

    # Initialize and fit the SplitCP conformal predictor
    splitcp = SplitCP(base_predictor, train=True, random_state=0)
    splitcp.fit(X=X_train, y=y_train, fit_ratio=0.5)

    # Compute prediction intervals and metrics on the test set using SplitCP
    y_pred, y_pred_lower, y_pred_upper = splitcp.predict(X_test, alpha=alpha)
    sharpness, coverage = evaluate_cp(X_test, y_test, splitcp, alpha)
    print(f"SplitCP - Average prediction intervals width (sharpness): {sharpness:.3f}")
    print(f"SplitCP - Average coverage: {coverage*100:.3f}%")

    # Plot the prediction intervals for SplitCP
    fig, axs = plt.subplots(3, 1, figsize=(10, 18))

    axs[0].scatter(np.arange(len(y_test)), y_test, label='True', color='blue', s=10, alpha=0.7)
    axs[0].fill_between(np.arange(len(y_test)), y_pred_lower, y_pred_upper, color='gray', alpha=0.5, label='Prediction interval')
    axs[0].plot(np.arange(len(y_test)), y_pred, color='red', label='Predicted mean', linewidth=1)
    axs[0].set_title('Split Conformal Prediction')
    axs[0].set_xlabel('Sample Number')
    axs[0].set_ylabel('Concrete Strength')
    axs[0].legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)

    # Initialize and fit the CVPlus conformal predictor
    cvplus = CVPlus(base_predictor, K=5, random_state=0)
    cvplus.fit(X=X_train, y=y_train)

    # Compute prediction intervals and metrics on the test set using CVPlus
    yi_pred, y_pred_lower, y_pred_upper = cvplus.predict(X_test, alpha=alpha)
    sharpness, coverage = evaluate_cp(X_test, y_test, cvplus, alpha)
    print(f"CVPlus - Average prediction intervals width (sharpness): {sharpness:.3f}")
    print(f"CVPlus - Average coverage: {coverage*100:.3f}%")

    # Plot the prediction intervals for CVPlus
    axs[1].scatter(np.arange(len(y_test)), y_test, label='True', color='blue', s=10, alpha=0.7)
    axs[1].fill_between(np.arange(len(y_test)), y_pred_lower, y_pred_upper, color='gray', alpha=0.5, label='Prediction interval')
    axs[1].plot(np.arange(len(y_test)), y_pred, color='red', label='Predicted mean', linewidth=1)
    axs[1].set_title('Cross Validation Plus')
    axs[1].set_xlabel('Sample Number ')
    axs[1].set_ylabel('Concrete Strength')
    axs[1].legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)

    # Split the training data into proper training (fit) set and calibration set
    X_fit, X_calib, y_fit, y_calib = train_test_split(
        X_train, y_train, test_size=0.5, random_state=0
    )

    # Fit the upper and lower quantile models
    upper_quantile_model = model_class(**model_params)
    lower_quantile_model = model_class(**model_params)

    _ = upper_quantile_model.fit(X_fit, y_fit)
    _ = lower_quantile_model.fit(X_fit, y_fit)

    # Wrap the upper and lower quantile models in a dual predictor
    dualpredictor = DualPredictor(
        [lower_quantile_model, upper_quantile_model], is_trained=[True, True]
    )

    # Initialize the CQR conformal predictor
    cqr = CQR(
        dualpredictor, train=False
    )  # train=False to use the pre-trained dual predictor

    # Compute nonconformity scores on the calibration set
    cqr.fit(X_calib=X_calib, y_calib=y_calib)

    # Compute prediction intervals and metrics on the test set
    y_pred, y_pred_lower, y_pred_upper = cqr.predict(X_test, alpha=alpha)
    sharpness, coverage = evaluate_cp(X_test, y_test, cqr, alpha)

    print(f"CQR - Average prediction intervals width (sharpness): {sharpness:.3f}")
    print(f"CQR - Average coverage: {coverage*100:.3f}%")

    # Plot the prediction intervals for CQR
    axs[2].scatter(np.arange(len(y_test)), y_test, label='True', color='blue', s=10, alpha=0.7)
    axs[2].fill_between(np.arange(len(y_test)), y_pred_lower, y_pred_upper, color='gray', alpha=0.5, label='Prediction interval')
    axs[2].plot(np.arange(len(y_test)), y_pred, color='red', label='Predicted mean', linewidth=1)
    axs[2].set_title('Conformalized Quantile Regression')
    axs[2].set_xlabel('Sample Number')
    axs[2].set_ylabel('Concrete Strength')
    axs[2].legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)

    plt.tight_layout()

    # Save all plots to a single Excel sheet
    if excel_file_path:
        save_plot_to_excel(fig, excel_file_path, 'conformal_predictions_PUNCC')

    plt.close(fig)

def save_plot_to_excel(fig, excel_file_path, sheet_name):
    with io.BytesIO() as buf:
        fig.savefig(buf, format='png')
        buf.seek(0)
        img = Image(buf)

        # Load the workbook and add a new sheet
        workbook = load_workbook(excel_file_path)
        if sheet_name in workbook.sheetnames:
            sheet_name += '_new'
        worksheet = workbook.create_sheet(title=sheet_name)

        # Add the image to the worksheet
        worksheet.add_image(img, 'A1')

        # Save the workbook
        workbook.save(excel_file_path)

In [16]:
def prediction_MAPIE_analysis(
    X_train, 
    y_train, 
    X_test, 
    y_test, 
    model_cls, 
    model_params, 
    excel_file_path=None,
    suptitle: str = "Prediction Intervals"
) -> None:
    # Initialize and fit the model with MAPIE using provided parameters
    mdl = model_cls(**model_params)
    mapie = MapieRegressor(mdl, method="plus", cv=KFold(n_splits=5, shuffle=True))
    mapie.fit(X_train, y_train)

    alpha = np.arange(0.05, 1, 0.05)
    y_train_pred, y_train_pis = mapie.predict(X_train, alpha=alpha)
    y_test_pred, y_test_pis = mapie.predict(X_test, alpha=alpha)

    # Visualization function
    def plot_predictionintervals(
        y_train,
        y_train_pred,
        y_train_pred_low,
        y_train_pred_high,
        y_test,
        y_test_pred,
        y_test_pred_low,
        y_test_pred_high,
        suptitle: str,
    ) -> None:
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 6))
        
        ax1.errorbar(
            x=y_train,
            y=y_train_pred,
            yerr=(np.abs(y_train_pred - y_train_pred_low), np.abs(y_train_pred_high - y_train_pred)),
            alpha=0.8,
            label="train",
            fmt=".",
        )
        ax1.errorbar(
            x=y_test,
            y=y_test_pred,
            yerr=(np.abs(y_test_pred - y_test_pred_low), np.abs(y_test_pred_high - y_test_pred)),
            alpha=0.8,
            label="test",
            fmt=".",
        )
        ax1.plot(
            [y_train.min(), y_train.max()],
            [y_train.min(), y_train.max()],
            color="gray",
            alpha=0.5,
        )
        ax1.set_xlabel("Actual Strength", fontsize=12)
        ax1.set_ylabel("Predicted Strength", fontsize=12)
        ax1.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        
        ax2.scatter(
            x=y_train, y=y_train_pred_high - y_train_pred_low, alpha=0.8, label="train", marker="."
        )
        ax2.scatter(x=y_test, y=y_test_pred_high - y_test_pred_low, alpha=0.8, label="test", marker=".")
        ax2.set_xlabel("Actual Strength", fontsize=12)
        ax2.set_ylabel("Interval width", fontsize=12)
        ax2.set_xscale("linear")
        ax2.set_ylim([0, np.max(y_test_pred_high - y_test_pred_low)*1.1])
        ax2.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        std_all = np.concatenate([
            y_train_pred_high - y_train_pred_low, y_test_pred_high - y_test_pred_low
        ])
        type_all = np.array(["train"] * len(y_train) + ["test"] * len(y_test))
        x_all = np.arange(len(std_all))
        order_all = np.argsort(std_all)
        std_order = std_all[order_all]
        type_order = type_all[order_all]
        ax3.scatter(
            x=x_all[type_order == "train"],
            y=std_order[type_order == "train"],
            alpha=0.8,
            label="train",
            marker=".",
        )
        ax3.scatter(
            x=x_all[type_order == "test"],
            y=std_order[type_order == "test"],
            alpha=0.8,
            label="test",
            marker=".",
        )
        ax3.set_xlabel("Order", fontsize=12)
        ax3.set_ylabel("Interval width", fontsize=12)
        ax3.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        ax1.set_title("Actual vs Predicted Strength")
        ax2.set_title("Prediction interval width vs Actual Strength")
        ax3.set_title("Ordered prediction interval width")
        plt.subplots_adjust(wspace=0.4, hspace=0.4)
        plt.suptitle(suptitle, size=20)

        # Save plot to Excel
        if excel_file_path:
            save_plot_to_excel(fig, excel_file_path, 'prediction_MAPIE_analysis_1')

        plt.close(fig)

    alpha_plot = int(np.where(alpha == 0.1)[0])
    plot_predictionintervals(
        y_train,
        y_train_pred,
        y_train_pis[:, 0, alpha_plot],
        y_train_pis[:, 1, alpha_plot],
        y_test,
        y_test_pred,
        y_test_pis[:, 0, alpha_plot],
        y_test_pis[:, 1, alpha_plot],
        suptitle,
    )

    # Comparison of the uncertainty quantification methods
    Params = TypedDict("Params", {"method": str, "cv": Union[int, Subsample]})
    STRATEGIES = {
        "naive": Params(method="naive"),
        "cv": Params(method="base", cv=5),
        "cv_plus": Params(method="plus", cv=5),
        "cv_minmax": Params(method="minmax", cv=5),
        "jackknife_plus_ab": Params(method="plus", cv=Subsample(n_resamplings=20)),
    }
    y_pred, y_pis, scores = {}, {}, {}
    for strategy, params in STRATEGIES.items():
        mapie = MapieRegressor(mdl, **params)
        mapie.fit(X_train, y_train)
        y_pred[strategy], y_pis[strategy] = mapie.predict(X_test, alpha=alpha)
        scores[strategy] = [
            regression_coverage_score(y_test, y_pis[strategy][:, 0, i], y_pis[strategy][:, 1, i])
            for i, _ in enumerate(alpha)
        ]
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.set_xlabel("Target coverage (1 - alpha)")
    ax.set_ylabel("Effective coverage")
    for strategy, params in STRATEGIES.items():
        ax.plot(1 - alpha, scores[strategy], label=strategy)
    plt.subplots_adjust(right=0.75)
    ax.plot([0, 1], [0, 1], ls="--", color="k")
    ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1), fontsize='small', borderaxespad=0.)

    # Save plot to Excel
    if excel_file_path:
        save_plot_to_excel(fig, excel_file_path, 'prediction_MAPIE_analysis_2')

    plt.close(fig)

def save_plot_to_excel(fig, excel_file_path, sheet_name):
    with io.BytesIO() as buf:
        fig.savefig(buf, format='png')
        buf.seek(0)
        img = Image(buf)

        # Load the workbook and add a new sheet
        workbook = load_workbook(excel_file_path)
        if sheet_name in workbook.sheetnames:
            sheet_name += '_new'
        worksheet = workbook.create_sheet(title=sheet_name)

        # Add the image to the worksheet
        worksheet.add_image(img, 'A1')

        # Save the workbook
        workbook.save(excel_file_path)

# **Hyperparameter Tuning using Autosampler Optuna**

In [17]:
# Suppress Optuna's verbose output
optuna.logging.set_verbosity(optuna.logging.WARNING)

def hyperparameter_tuning_optuna(X_train, y_train, X_test, y_test):
    def objective(trial, model_name, model_class, param_space):
        # Suggest hyperparameters
        params = {}
        for key, values in param_space.items():
            if isinstance(values, list):
                params[key] = trial.suggest_categorical(key, values)
            elif isinstance(values, tuple):
                if len(values) == 2:
                    params[key] = trial.suggest_float(key, values[0], values[1])
                elif len(values) == 3 and isinstance(values[2], bool) and values[2]:
                    params[key] = trial.suggest_int(key, values[0], values[1])
                else:
                    raise ValueError(f"Invalid parameter range for {key}")
            else:
                raise ValueError(f"Invalid parameter type for {key}")

        # Create a new instance of the model
        model = model_class()
        model.set_params(**params)

        # Ensure the input arrays are writable
        X_train_copy = np.array(X_train, copy=True)
        y_train_copy = np.array(y_train, copy=True)
        X_test_copy = np.array(X_test, copy=True)
        y_test_copy = np.array(y_test, copy=True)

        # Train model
        model.fit(X_train_copy, y_train_copy)

        # Predict
        y_pred = model.predict(X_test_copy)

        # Calculate MSE
        mse = mean_squared_error(y_test_copy, y_pred)

        return mse

    models = {
        'XGBoost': (XGBRegressor, {
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'max_depth': [3, 5, 7],
            'n_estimators': [100, 200, 300, 400, 500],
            'min_child_weight': [1, 3, 5],
            'gamma': [0, 0.1, 0.5, 1],
            'subsample': [0.5, 0.6, 0.7, 0.8, 0.9],
            'colsample_bytree': [0.5, 0.7, 0.9]
        }),
        'GPBoost': (GPBoostRegressor, {
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'max_depth': [3, 5, 7],
            'n_estimators': [100, 200, 300, 400, 500],
            'num_leaves': [15, 31, 63],
            'min_child_samples': [1, 5, 10],
            'subsample': [0.5, 0.6, 0.7, 0.8, 0.9],
            'colsample_bytree': [0.5, 0.7, 0.9],
            'alpha': [0.1, 0.5, 1.0],
            'lambda': [0.1, 0.5, 1.0],
            'verbose': [-1]
        }),
        'LightGBM': (LGBMRegressor, {
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'max_depth': [3, 5, 7],
            'n_estimators': [100, 200, 300, 400, 500],
            'num_leaves': [15, 31, 63],
            'min_child_samples': [1, 5, 10],
            'subsample': [0.5, 0.6, 0.7, 0.8, 0.9],
            'colsample_bytree': [0.5, 0.7, 0.9],
            'verbose': [-1]
        }),
        'NGBoost': (NGBRegressor, {
            'n_estimators': [100, 200, 300, 400, 500],
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'natural_gradient': [True, False],
            'minibatch_frac': [0.5, 0.7],
            'verbose': [0]
        })
    }

    pruners = [
        optuna.pruners.MedianPruner(),
        optuna.pruners.NopPruner(),
        optuna.pruners.PatientPruner(optuna.pruners.MedianPruner(), patience=3),
        optuna.pruners.PercentilePruner(25.0),
        optuna.pruners.SuccessiveHalvingPruner(),
        optuna.pruners.HyperbandPruner(),
        optuna.pruners.ThresholdPruner(lower=0.1),
        optuna.pruners.WilcoxonPruner()
    ]

    best_scores = {}
    for model_name, (model_class, param_space) in models.items():
        for pruner in pruners:
            pruner_name = pruner.__class__.__name__
            print(f"Running Optuna for {model_name} with {pruner_name}...")
            try:
                # Use AutoSampler from OptunaHub
                sampler = optunahub.load_module("samplers/auto_sampler").AutoSampler()
                study = optuna.create_study(direction='minimize', sampler=sampler, pruner=pruner)
                study.optimize(lambda trial: objective(trial, model_name, model_class, param_space), n_trials=50)

                best_params = study.best_params
                best_model = model_class().set_params(**best_params)
                best_model.fit(X_train, y_train)
                y_pred = best_model.predict(X_test)

                # Calculate MSE
                mse = mean_squared_error(y_test, y_pred)

                # Calculate RMSE
                rmse = np.sqrt(mse)

                # Calculate correlation coefficient
                corr_coef = np.corrcoef(y_test, y_pred)[0, 1]

                best_scores[(model_name, pruner_name)] = {
                    'best_score': mse,
                    'best_params': best_params,
                    'test_mse': mse,
                    'test_rmse': rmse,
                    'test_corr_coef': corr_coef,
                    'pruner': pruner_name
                }
                print(f"Best MSE for {model_name} with {pruner_name}: {mse} with params: {best_params}")
                print(f"Best RMSE for {model_name} with {pruner_name}: {rmse}")
                print(f"Correlation Coefficient for {model_name} with {pruner_name}: {corr_coef}")
            except Exception as e:
                print(f"Failed to run Optuna for {model_name} with {pruner_name}. Error: {e}")

    # Find the model with the best test score
    if best_scores:
        best_model_name, best_pruner_name = min(best_scores, key=lambda k: best_scores[k]['test_mse'])
        best_model_info = best_scores[(best_model_name, best_pruner_name)]
        print(f"\nBest model on test data: {best_model_name} with {best_pruner_name}")
        print(f"Test MSE: {best_model_info['test_mse']}")
        print(f"Test RMSE: {best_model_info['test_rmse']}")
        print(f"Correlation Coefficient: {best_model_info['test_corr_coef']}")
        print(f"Best Parameters: {best_model_info['best_params']}")
        print(f"Pruner Used: {best_model_info['pruner']}")
    else:
        print("No valid model configurations found.")

    return best_scores

# Example usage:
best_scores_autosampler = hyperparameter_tuning_optuna(X_train, y_train, X_test, y_test)

Running Optuna for XGBoost with MedianPruner...
Best MSE for XGBoost with MedianPruner: 7.7183279656157175 with params: {'learning_rate': 0.05, 'max_depth': 7, 'n_estimators': 500, 'min_child_weight': 5, 'gamma': 0.5, 'subsample': 0.8, 'colsample_bytree': 0.9}
Best RMSE for XGBoost with MedianPruner: 2.7781878924247936
Correlation Coefficient for XGBoost with MedianPruner: 0.9605510331713474
Running Optuna for XGBoost with NopPruner...
Best MSE for XGBoost with NopPruner: 7.898137975836169 with params: {'learning_rate': 0.05, 'max_depth': 7, 'n_estimators': 300, 'min_child_weight': 3, 'gamma': 0, 'subsample': 0.6, 'colsample_bytree': 0.7}
Best RMSE for XGBoost with NopPruner: 2.8103626057568034
Correlation Coefficient for XGBoost with NopPruner: 0.9596328244094964
Running Optuna for XGBoost with PatientPruner...
Best MSE for XGBoost with PatientPruner: 7.677634854391444 with params: {'learning_rate': 0.05, 'max_depth': 7, 'n_estimators': 400, 'min_child_weight': 5, 'gamma': 0.1, 'subsa

In [18]:
best_scores_autosampler

{('XGBoost', 'MedianPruner'): {'best_score': 7.7183279656157175,
  'best_params': {'learning_rate': 0.05,
   'max_depth': 7,
   'n_estimators': 500,
   'min_child_weight': 5,
   'gamma': 0.5,
   'subsample': 0.8,
   'colsample_bytree': 0.9},
  'test_mse': 7.7183279656157175,
  'test_rmse': 2.7781878924247936,
  'test_corr_coef': 0.9605510331713474,
  'pruner': 'MedianPruner'},
 ('XGBoost', 'NopPruner'): {'best_score': 7.898137975836169,
  'best_params': {'learning_rate': 0.05,
   'max_depth': 7,
   'n_estimators': 300,
   'min_child_weight': 3,
   'gamma': 0,
   'subsample': 0.6,
   'colsample_bytree': 0.7},
  'test_mse': 7.898137975836169,
  'test_rmse': 2.8103626057568034,
  'test_corr_coef': 0.9596328244094964,
  'pruner': 'NopPruner'},
 ('XGBoost', 'PatientPruner'): {'best_score': 7.677634854391444,
  'best_params': {'learning_rate': 0.05,
   'max_depth': 7,
   'n_estimators': 400,
   'min_child_weight': 5,
   'gamma': 0.1,
   'subsample': 0.8,
   'colsample_bytree': 0.9},
  'test_

# **Uncertainity evaluation with Lightgbm**

In [19]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_LGBM_df = pd.DataFrame()


# Train a model for each quantile and make predictions using LightGBM
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = LGBMRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_LGBM_df[q] = predictions

# Add actual target values to the DataFrame
predictions_LGBM_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_LGBM_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  26.693332  28.573971  31.640630  31.189738  31.324406  32.782719   
1  29.940465  31.996373  32.870810  32.472396  34.096739  36.957128   
2  32.809426  33.249756  33.773434  33.157586  31.628715  33.433446   
3  31.609382  33.305136  33.108953  32.061176  33.750094  36.640884   
4  39.991140  40.515910  42.466540  41.791791  40.645095  39.469960   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  32.219428  32.649583  32.757093  33.738667  34.134266  35.008787   
1  36.287200  36.370060  36.165859  37.553035  36.392291  36.722219   
2  35.014158  34.980493  34.615684  35.602714  35.005381  35.978385   
3  36.069387  36.173710  36.425075  37.032481  36.831628  36.711082   
4  41.651530  42.634563  45.063739  50.632540  50.870090  57.244711   

        0.95  Actual  
0  40.035519    34.0  
1  36.907705    36.0  
2  38.920611    36.0  
3  36.852352    35.0  
4  60.336328    44.6  


In [20]:
quantile_regression_print(predictions_LGBM_df,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_LightGBM.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 58.95%


In [21]:
evaluation_features_print(predictions_LGBM_df,x_test,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_LightGBM.xlsx")

In [22]:
evaluate_uncertainity_matrix(predictions_LGBM_df, quantiles, r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_LightGBM.xlsx", model_name='LightGBM')

Coverage level: 60%, Interval: [0.2, 0.8], Average Interval Length: 3.1308
Coverage level: 80%, Interval: [0.1, 0.9], Average Interval Length: 5.1999
Coverage level: 90%, Interval: [0.05, 0.95], Average Interval Length: 7.7471
Average Negative Log-Likelihood (NLL): 15.3198
Average Continuous Ranked Probability Score (CRPS): 1.6919


{'empirical_quantiles': [0.16842105263157894,
  0.21052631578947367,
  0.2631578947368421,
  0.28421052631578947,
  0.3473684210526316,
  0.49473684210526314,
  0.5052631578947369,
  0.5684210526315789,
  0.631578947368421,
  0.7052631578947368,
  0.7157894736842105,
  0.7894736842105263,
  0.7578947368421053],
 'average_interval_lengths': [3.1307553887401234,
  5.199864321030318,
  7.7471193108139795],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': 15.31983925252315,
 'average_crps': 1.6918562160369908,
 'nll_values': array([ 2.41544267e+00,  1.67855365e+00,  1.67907539e+00,  1.61014061e+00,
         2.85463047e+00,  2.11214677e+00,  5.92533436e+00,  1.32845748e+00,
         2.87828719e+00,  1.81010764e+00,  1.29918090e+00, -1.01145555e-01,
         1.51858653e+00,  2.88267476e+00,  2.93925073e+00,  1.48297463e+00,
         3.00478042e+00,  1.05665877e+00,  2.27716134e+00,  1.26354058e+00,
         2.43591825e+00,  1.77929391e+00,  1.88539441e+00,  2.65313863e+00,
         1.468

In [23]:
calibrate_and_plot_intervals(predictions_LGBM_df, r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_LightGBM.xlsx")

The empirical coverage before calibration is: 54.17%
The empirical coverage after calibration is: 93.75%


In [24]:
plot_conformal_intervals_to_excel(predictions_LGBM_df,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_LightGBM.xlsx")

In [25]:
conformal_predictions_MAPIE(LGBMRegressor, best_params, X_train, y_train, X_test, y_test, "LightGBM",r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_LightGBM.xlsx")

Strategy: Split Conformal Prediction
  RMSE: 2.8743
  Interval Width: 22.1254
  Coverage: 61.78%
  CWC: 30.5809
  SSC: 497.7953

Strategy: Cross Validation +
  RMSE: 2.4989
  Interval Width: 18.5778
  Coverage: 54.20%
  CWC: 27.0856
  SSC: 352.4095

Strategy: Jackknife+ after Bootstrapping
  RMSE: 2.4989
  Interval Width: 8.9717
  Coverage: 28.59%
  CWC: 15.3786
  SSC: 88.1851



In [26]:
conformal_predictions_PUNCC(X_train, y_train, X_test, y_test, best_scores_autosampler, LGBMRegressor,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_LightGBM.xlsx")

SplitCP - Average prediction intervals width (sharpness): 12.510
SplitCP - Average coverage: 92.632%
CVPlus - Average prediction intervals width (sharpness): 12.609
CVPlus - Average coverage: 97.895%
CQR - Average prediction intervals width (sharpness): 23.500
CQR - Average coverage: 97.895%


In [27]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')
prediction_MAPIE_analysis(X_train, y_train, X_test, y_test, LGBMRegressor, best_params, r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_LightGBM.xlsx", "LightGBM Prediction Intervals")

# **Uncertainity evaluation with QXGBoost**

In [28]:
# Get the best hyperparameters for XGBRegressor
best_paramst = get_best_model_params(best_scores_autosampler, 'XGBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3, 
             0.4, 0.5, 0.6, 0.7, 0.8, 
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_XGB_df = pd.DataFrame()

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Adjust according to your XGBoost version
    params.update({
        'objective': 'reg:quantileerror',  # Use the correct objective
        'quantile_alpha': q,                         # Use 'alpha' to set the quantile level
        'random_state': 42
    })
    
    # Initialize the model with the updated parameters
    model = XGBRegressor(**params)
    
    # Fit the model
    model.fit(X_train, y_train)
    
    # Store the model
    models_quantile[q] = model
    
    # Make predictions
    predictions = model.predict(X_test)
    
    # Add predictions to the DataFrame
    predictions_XGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_XGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_XGB_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  28.340527  30.692156  31.237335  32.380291  33.029362  33.023781   
1  34.678841  34.844353  34.873734  35.142536  35.458199  36.484852   
2  32.959015  34.972267  34.962811  34.706135  34.836849  34.717464   
3  32.136936  33.161816  33.587151  33.256100  34.647911  34.339241   
4  42.533329  45.495457  45.684875  43.627377  44.313915  44.474831   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  32.486500  32.826603  33.057045  35.698174  35.752575  38.700626   
1  36.847328  36.510399  36.221775  37.499596  36.926537  36.962059   
2  34.908512  34.884510  34.778019  35.077534  35.888458  38.497002   
3  36.131443  35.692238  35.132687  35.667377  36.388336  36.700558   
4  43.551380  42.509464  45.263851  43.648811  45.344147  47.248852   

        0.95  Actual  
0  41.911606    34.0  
1  37.612209    36.0  
2  41.458843    36.0  
3  38.173954    35.0  
4  46.535118    44.6  


In [29]:
quantile_regression_print(predictions_XGB_df,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_XGBoost.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 65.26%


In [30]:
evaluation_features_print(predictions_XGB_df,x_test,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_XGBoost.xlsx")

In [31]:
evaluate_uncertainity_matrix(predictions_XGB_df, quantiles, r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_XGBoost.xlsx", model_name='XGBoost')

Coverage level: 60%, Interval: [0.2, 0.8], Average Interval Length: 2.7694
Coverage level: 80%, Interval: [0.1, 0.9], Average Interval Length: 5.7283
Coverage level: 90%, Interval: [0.05, 0.95], Average Interval Length: 9.0373
Average Negative Log-Likelihood (NLL): 620397762422.9172
Average Continuous Ranked Probability Score (CRPS): 1.8849


{'empirical_quantiles': [0.14736842105263157,
  0.22105263157894736,
  0.28421052631578947,
  0.3263157894736842,
  0.3894736842105263,
  0.4,
  0.49473684210526314,
  0.5578947368421052,
  0.5894736842105263,
  0.6210526315789474,
  0.6736842105263158,
  0.7263157894736842,
  0.8],
 'average_interval_lengths': [2.7693903, 5.7282624, 9.03735],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': 620397762422.9172,
 'average_crps': 1.8849183950034885,
 'nll_values': array([2.40338183e+00, 1.25578807e+00, 1.95741409e+00, 1.71611636e+00,
        1.48642720e+00, 1.95737470e+00, 6.18016659e+00, 3.69566954e-01,
        3.12637822e+00, 1.91854897e+00, 2.23136075e-01, 1.77595362e+00,
        4.64112816e+00, 2.95277474e+00, 3.18932031e+00, 2.14729075e+00,
        3.61053859e+00, 1.93909158e+00, 2.52138005e+00, 1.53216351e+00,
        2.46626861e+00, 1.91626985e+00, 2.25312829e+00, 2.01539857e+00,
        1.54747314e+00, 1.73611135e+00, 2.26947567e+00, 1.43074741e+01,
        2.09793270e+00, 1.4

In [32]:
calibrate_and_plot_intervals(predictions_XGB_df, r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_XGBoost.xlsx")

The empirical coverage before calibration is: 66.67%
The empirical coverage after calibration is: 87.50%


In [33]:
plot_conformal_intervals_to_excel(predictions_XGB_df, r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_XGBoost.xlsx")

In [34]:
conformal_predictions_MAPIE(XGBRegressor, best_params, X_train, y_train, X_test, y_test, "XGBoost",r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_XGBoost.xlsx")

Strategy: Split Conformal Prediction
  RMSE: 3.2173
  Interval Width: 26.2726
  Coverage: 69.86%
  CWC: 34.1908
  SSC: 700.6025

Strategy: Cross Validation +
  RMSE: 2.8890
  Interval Width: 18.0542
  Coverage: 53.12%
  CWC: 26.5182
  SSC: 334.5437

Strategy: Jackknife+ after Bootstrapping
  RMSE: 2.8890
  Interval Width: 8.4872
  Coverage: 27.41%
  CWC: 14.6478
  SSC: 81.5046



In [35]:
conformal_predictions_PUNCC(X_train, y_train, X_test, y_test, best_scores_autosampler, XGBRegressor,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_XGBoost.xlsx")

SplitCP - Average prediction intervals width (sharpness): 14.054
SplitCP - Average coverage: 92.632%
CVPlus - Average prediction intervals width (sharpness): 12.679
CVPlus - Average coverage: 95.789%
CQR - Average prediction intervals width (sharpness): 22.252
CQR - Average coverage: 94.737%


In [36]:
best_params = get_best_model_params(best_scores_autosampler, 'XGBoost')
prediction_MAPIE_analysis(X_train, y_train, X_test, y_test, XGBRegressor, best_params,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_XGBoost.xlsx", "XGBM Prediction Intervals")

# **Uncertainity evaluation with GPBoost**

In [37]:
# Get the best hyperparameters for GPBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GPBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'alpha', 'quantile_alpha', 'boosting_type', 'metric', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()
    
    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)
    
    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })
    
    # Initialize the model with the updated parameters
    model = GPBoostRegressor(**params)
    
    # Fit the model
    model.fit(X_train, y_train)
    
    # Store the model
    models_quantile[q] = model
    
    # Make predictions
    predictions = model.predict(X_test)
    
    # Add predictions to the DataFrame
    predictions_GPBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GPBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GPBoost_df.head())

[GPBoost] [Warning] lambda_l2 is set with reg_lambda=0.0, will be overridden by lambda=0.1. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with reg_lambda=0.0, will be overridden by lambda=0.1. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with reg_lambda=0.0, will be overridden by lambda=0.1. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with reg_lambda=0.0, will be overridden by lambda=0.1. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with reg_lambda=0.0, will be overridden by lambda=0.1. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with reg_lambda=0.0, will be overridden by lambda=0.1. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with reg_lambda=0.0, will be overridden by lambda=0.1. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with reg_lambda=0.0, will be overridden by lambda=0.1. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is

In [38]:
quantile_regression_print(predictions_GPBoost_df,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_GPBoost.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 71.58%


In [39]:
evaluation_features_print(predictions_GPBoost_df,x_test,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_GPBoost.xlsx")

In [40]:
evaluate_uncertainity_matrix(predictions_GPBoost_df, quantiles,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_GPBoost.xlsx", model_name='GPBoost')

Coverage level: 60%, Interval: [0.2, 0.8], Average Interval Length: 3.3077
Coverage level: 80%, Interval: [0.1, 0.9], Average Interval Length: 6.0893
Coverage level: 90%, Interval: [0.05, 0.95], Average Interval Length: 9.9997
Average Negative Log-Likelihood (NLL): 2.4379
Average Continuous Ranked Probability Score (CRPS): 1.6890


{'empirical_quantiles': [0.11578947368421053,
  0.16842105263157894,
  0.25263157894736843,
  0.2736842105263158,
  0.4105263157894737,
  0.4631578947368421,
  0.5157894736842106,
  0.6,
  0.6842105263157895,
  0.7684210526315789,
  0.7894736842105263,
  0.8,
  0.8315789473684211],
 'average_interval_lengths': [3.30772274455229,
  6.089291696126236,
  9.999740733984847],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': 2.4378841817116443,
 'average_crps': 1.6890045992807399,
 'nll_values': array([ 2.39711447,  2.13938623,  1.85081604,  2.00094879,  2.84231937,
         2.35161296,  4.92147687,  1.64874579,  2.9617386 ,  2.18391427,
         1.48532505,  1.21291323,  1.5461363 ,  3.04477766,  3.13564639,
         1.88787758,  3.12470303,  1.56033599,  2.37785327,  1.58919397,
         1.92925907,  1.62298097,  2.3129909 ,  1.99009695,  1.28035966,
         1.94002311,  2.45491284,  3.93556906,  1.50305105,  1.92804868,
         1.69562819,  4.77208786,  2.09861008,  1.60312897,  2.7

In [41]:
calibrate_and_plot_intervals(predictions_GPBoost_df,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_GPBoost.xlsx")

The empirical coverage before calibration is: 72.92%
The empirical coverage after calibration is: 89.58%


In [42]:
plot_conformal_intervals_to_excel(predictions_GPBoost_df,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_GPBoost.xlsx")

In [43]:
conformal_predictions_MAPIE(GPBoostRegressor, best_params, X_train, y_train, X_test, y_test, "GPBoost",r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_GPBoost.xlsx")

[GPBoost] [Warning] lambda_l2 is set with reg_lambda=0.0, will be overridden by lambda=0.1. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with reg_lambda=0.0, will be overridden by lambda=0.1. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with reg_lambda=0.0, will be overridden by lambda=0.1. Current value: lambda_l2=0.1
Strategy: Split Conformal Prediction
  RMSE: 2.8530
  Interval Width: 22.4391
  Coverage: 62.30%
  CWC: 30.8975
  SSC: 511.6512

Strategy: Cross Validation +
  RMSE: 2.5297
  Interval Width: 19.4410
  Coverage: 55.48%
  CWC: 28.0962
  SSC: 385.0487

Strategy: Jackknife+ after Bootstrapping
  RMSE: 2.5297
  Interval Width: 8.8999
  Coverage: 28.73%
  CWC: 15.2428
  SSC: 87.0421



In [44]:
conformal_predictions_PUNCC(X_train, y_train, X_test, y_test, best_scores_autosampler, GPBoostRegressor,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_GPBoost.xlsx")

Finished loading model, total used 100 iterations
SplitCP - Average prediction intervals width (sharpness): 17.856
SplitCP - Average coverage: 93.684%
Finished loading model, total used 100 iterations
Finished loading model, total used 100 iterations
Finished loading model, total used 100 iterations
Finished loading model, total used 100 iterations
Finished loading model, total used 100 iterations
Finished loading model, total used 100 iterations
Finished loading model, total used 100 iterations
Finished loading model, total used 100 iterations
Finished loading model, total used 100 iterations
Finished loading model, total used 100 iterations
CVPlus - Average prediction intervals width (sharpness): 16.267
CVPlus - Average coverage: 94.737%
Finished loading model, total used 100 iterations
Finished loading model, total used 100 iterations
CQR - Average prediction intervals width (sharpness): 29.261
CQR - Average coverage: 95.789%


In [45]:
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')
prediction_MAPIE_analysis(X_train, y_train, X_test, y_test, GPBoostRegressor, best_params, r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_GPBoost.xlsx", "GPBoost Prediction Intervals")

[GPBoost] [Warning] lambda_l2 is set with reg_lambda=0.0, will be overridden by lambda=0.1. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with reg_lambda=0.0, will be overridden by lambda=0.1. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with reg_lambda=0.0, will be overridden by lambda=0.1. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with reg_lambda=0.0, will be overridden by lambda=0.1. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with reg_lambda=0.0, will be overridden by lambda=0.1. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with reg_lambda=0.0, will be overridden by lambda=0.1. Current value: lambda_l2=0.1


# **Uncertainity evaluation with NGBoost**

In [46]:
# Correct best_params assignment
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')

# DataFrame to store predictions
predictions_NGB_df = pd.DataFrame()

# Remove 'Score' and 'Dist' if present in best_params
for key in ['Score', 'Dist']:
    best_params.pop(key, None)

# Update params with model settings
best_params.update({
    'Dist': Normal,
    'Score': LogScore,
    'random_state': 42,
    'verbose': True
})

# Initialize and fit the NGBoost model once
model = NGBRegressor(**best_params)

# Fit the model
model.fit(X_train, y_train)

# Make predictions
pred_dist = model.pred_dist(X_test)

# Extract the mean (mu) and standard deviation (sigma) of the predicted distributions
mu = pred_dist.loc    # Mean predictions
sigma = pred_dist.scale  # Standard deviation predictions

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# For each quantile, compute predictions
for q in quantiles:
    # Calculate the z-score for the desired quantile
    z = norm.ppf(q)
    
    # Compute the quantile prediction per sample
    predictions = mu + z * sigma
    
    # Add predictions to the DataFrame
    predictions_NGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_NGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_NGB_df.head())

[iter 0] loss=3.6984 val_loss=0.0000 scale=1.0000 norm=8.0525
[iter 100] loss=1.2013 val_loss=0.0000 scale=1.0000 norm=0.9571
[iter 200] loss=0.7268 val_loss=0.0000 scale=0.5000 norm=0.3576
        0.05        0.1       0.15        0.2        0.3        0.4  \
0  32.672511  32.831940  32.939506  33.024996  33.164203  33.283150   
1  35.876839  35.976330  36.043455  36.096804  36.183675  36.257903   
2  33.100316  33.396880  33.596971  33.755997  34.014945  34.236207   
3  36.025818  36.162050  36.253965  36.327016  36.445968  36.547608   
4  43.746594  43.800713  43.837227  43.866248  43.913503  43.953880   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  33.394327  33.505504  33.624451  33.763658  33.849148  33.956714   
1  36.327282  36.396661  36.470889  36.557760  36.611109  36.678235   
2  34.443015  34.649823  34.871084  35.130033  35.289058  35.489149   
3  36.642608  36.737609  36.839249  36.958201  37.031252  37.123167   
4  43.991620  44.029361  44.

In [47]:
quantile_regression_print(predictions_NGB_df,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_NGBoost.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 35.79%


In [48]:
evaluation_features_print(predictions_NGB_df,x_test,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_NGBoost.xlsx")

In [49]:
evaluate_uncertainity_matrix(predictions_NGB_df, quantiles,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_NGBoost.xlsx", model_name='NGBoost')

Coverage level: 60%, Interval: [0.2, 0.8], Average Interval Length: 0.8710
Coverage level: 80%, Interval: [0.1, 0.9], Average Interval Length: 1.3263
Coverage level: 90%, Interval: [0.05, 0.95], Average Interval Length: 1.7023
Average Negative Log-Likelihood (NLL): 286.9789
Average Continuous Ranked Probability Score (CRPS): 1.9395


{'empirical_quantiles': [0.2736842105263158,
  0.3157894736842105,
  0.35789473684210527,
  0.37894736842105264,
  0.4105263157894737,
  0.42105263157894735,
  0.43157894736842106,
  0.4421052631578947,
  0.45263157894736844,
  0.5052631578947369,
  0.5368421052631579,
  0.5684210526315789,
  0.631578947368421],
 'average_interval_lengths': [0.8710321471518339,
  1.3263360848005492,
  1.7023339351262905],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': 286.97890168035127,
 'average_crps': 1.939531250989865,
 'nll_values': array([ 1.04776427e+00,  3.37913382e-01,  2.53498405e+00,  9.53244764e+00,
         7.35451472e+00,  8.93809233e-02,  2.72786339e+03,  4.79137153e-01,
         1.62163571e+01,  4.20903759e-01,  2.50638518e-01,  6.69665238e-01,
         5.23528664e+00,  1.62079160e+00,  3.46307397e+01,  1.04634495e+00,
         7.81748306e-01,  9.61898209e-01,  5.87463777e-01,  1.25479628e+00,
         5.58975176e+01,  2.40040883e+00,  1.84101570e+00,  4.02359197e+00,
         1.8

In [50]:
calibrate_and_plot_intervals(predictions_NGB_df,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_NGBoost.xlsx")

The empirical coverage before calibration is: 35.42%
The empirical coverage after calibration is: 89.58%


In [51]:
plot_conformal_intervals_to_excel(predictions_NGB_df,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_NGBoost.xlsx")

In [52]:
conformal_predictions_MAPIE(NGBRegressor, best_params, X_train, y_train, X_test, y_test, "NGBoost",r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_NGBoost.xlsx")

[iter 0] loss=3.7392 val_loss=0.0000 scale=1.0000 norm=8.2065
[iter 100] loss=1.0651 val_loss=0.0000 scale=1.0000 norm=0.8698
[iter 200] loss=0.6073 val_loss=0.0000 scale=0.2500 norm=0.1746
[iter 0] loss=3.7392 val_loss=0.0000 scale=1.0000 norm=8.2065
[iter 100] loss=0.9878 val_loss=0.0000 scale=1.0000 norm=0.8560
[iter 200] loss=0.7011 val_loss=0.0000 scale=0.5000 norm=0.3554
[iter 0] loss=3.6984 val_loss=0.0000 scale=1.0000 norm=8.0525
[iter 100] loss=1.1629 val_loss=0.0000 scale=1.0000 norm=0.9069
[iter 200] loss=0.6645 val_loss=0.0000 scale=0.2500 norm=0.1766
[iter 0] loss=3.7321 val_loss=0.0000 scale=1.0000 norm=8.1557
[iter 100] loss=1.1084 val_loss=0.0000 scale=1.0000 norm=0.8626
[iter 200] loss=0.5898 val_loss=0.0000 scale=1.0000 norm=0.6434
[iter 0] loss=3.7323 val_loss=0.0000 scale=1.0000 norm=8.1829
[iter 100] loss=1.1139 val_loss=0.0000 scale=1.0000 norm=0.8976
[iter 200] loss=0.5578 val_loss=0.0000 scale=1.0000 norm=0.6467
[iter 0] loss=3.6442 val_loss=0.0000 scale=1.0000 

In [53]:
conformal_predictions_PUNCC(X_train, y_train, X_test, y_test, best_scores_autosampler, NGBRegressor,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_NGBoost.xlsx")

[iter 0] loss=3.7185 val_loss=0.0000 scale=1.0000 norm=8.0786
[iter 100] loss=3.1871 val_loss=0.0000 scale=2.0000 norm=9.1519
[iter 200] loss=2.7379 val_loss=0.0000 scale=1.0000 norm=3.1132
[iter 300] loss=2.4646 val_loss=0.0000 scale=1.0000 norm=2.5738
[iter 400] loss=2.2767 val_loss=0.0000 scale=2.0000 norm=4.5147
[iter 0] loss=3.7001 val_loss=0.0000 scale=1.0000 norm=7.8643
[iter 100] loss=3.1923 val_loss=0.0000 scale=1.0000 norm=4.8492
[iter 200] loss=2.7182 val_loss=0.0000 scale=2.0000 norm=5.9965
[iter 300] loss=2.3327 val_loss=0.0000 scale=1.0000 norm=2.1939
[iter 400] loss=2.0801 val_loss=0.0000 scale=1.0000 norm=1.8350
SplitCP - Average prediction intervals width (sharpness): 12.382
SplitCP - Average coverage: 82.105%
[iter 0] loss=3.6768 val_loss=0.0000 scale=1.0000 norm=7.7253
[iter 100] loss=3.0959 val_loss=0.0000 scale=1.0000 norm=4.2715
[iter 200] loss=2.6413 val_loss=0.0000 scale=1.0000 norm=2.9140
[iter 300] loss=2.3708 val_loss=0.0000 scale=1.0000 norm=2.4185
[iter 400

In [54]:
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')
prediction_MAPIE_analysis(X_train, y_train, X_test, y_test, NGBRegressor, best_params,r"C:\Users\Danesh\Desktop\Concrete\Uncertainity_NGBoost.xlsx", "NGBoost Prediction Intervals")

[iter 0] loss=3.6984 val_loss=0.0000 scale=1.0000 norm=8.0525
[iter 100] loss=1.2172 val_loss=0.0000 scale=1.0000 norm=0.9477
[iter 200] loss=0.6810 val_loss=0.0000 scale=0.5000 norm=0.3499
[iter 0] loss=3.7468 val_loss=0.0000 scale=1.0000 norm=8.0531
[iter 100] loss=1.0519 val_loss=0.0000 scale=1.0000 norm=0.8147
[iter 200] loss=0.5358 val_loss=0.0000 scale=0.5000 norm=0.3195
[iter 0] loss=3.6708 val_loss=0.0000 scale=1.0000 norm=7.7751
[iter 100] loss=0.8905 val_loss=0.0000 scale=1.0000 norm=0.7800
[iter 200] loss=0.3445 val_loss=0.0000 scale=0.5000 norm=0.3095
[iter 0] loss=3.6145 val_loss=0.0000 scale=1.0000 norm=7.4139
[iter 100] loss=1.1671 val_loss=0.0000 scale=1.0000 norm=0.9181
[iter 200] loss=0.5323 val_loss=0.0000 scale=0.5000 norm=0.3212
[iter 0] loss=3.7088 val_loss=0.0000 scale=1.0000 norm=8.0272
[iter 100] loss=1.0887 val_loss=0.0000 scale=1.0000 norm=0.8597
[iter 200] loss=0.5504 val_loss=0.0000 scale=0.5000 norm=0.3242
[iter 0] loss=3.6961 val_loss=0.0000 scale=1.0000 